In [3]:
# ============================================================
# VideoMAE Fine-tuning — HPC code-server version
# Diadaptasi dari kode kating, path & dependencies disesuaikan
# untuk lingkungan /home/coder (bukan Google Colab)
# ============================================================

# HAPUS: !pip install (jalankan manual di terminal dulu)
# pip install pytorchvideo transformers evaluate scikit-learn imageio

import sys, os, types, json
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Fix torchvision compatibility (sama seperti kode kating)
from torchvision.transforms.functional import rgb_to_grayscale
functional_tensor = types.ModuleType("torchvision.transforms.functional_tensor")
functional_tensor.rgb_to_grayscale = rgb_to_grayscale
sys.modules["torchvision.transforms.functional_tensor"] = functional_tensor

from pytorchvideo.data import LabeledVideoDataset, make_clip_sampler
from pytorchvideo.transforms import (
    ApplyTransformToKey, Normalize, RandomShortSideScale,
    ShortSideScale, UniformTemporalSubsample,
)
from torchvision.transforms import (
    Compose, Lambda, RandomCrop, RandomHorizontalFlip, Resize,
)
from transformers import (
    VideoMAEImageProcessor, VideoMAEForVideoClassification,
    TrainingArguments, Trainer,
)
import evaluate
import pytorchvideo.data

# -------------------------
# PATH CONFIG — sesuaikan di sini saja
# -------------------------
ROOT_PATH    = "/home/coder/data_skripsi/dataset_teh_mutia"
OUTPUT_DIR   = "/home/coder/output_model/videomae_zoomodel"
LOG_DIR      = "/home/coder/output_model/videomae_zoomodel/logs"

# Pakai pretrained dari HuggingFace — tidak butuh checkpoint kating
MODEL_CHECKPOINT = "MCG-NJU/videomae-base"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR,    exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# -------------------------
# Load dataset dari folder struktur train/val/test
# -------------------------
def create_df_from_split(split):
    split_dir = os.path.join(ROOT_PATH, split)
    paths, labels = [], []
    for label_name in sorted(os.listdir(split_dir)):
        label_dir = os.path.join(split_dir, label_name)
        if not os.path.isdir(label_dir):
            continue
        for fname in os.listdir(label_dir):
            if fname.lower().endswith((".mp4", ".avi")):
                paths.append(os.path.join(label_dir, fname))
                labels.append(label_name)
    return pd.DataFrame({"Path": paths, "Label": labels})

train_df = create_df_from_split("train")
val_df   = create_df_from_split("val")
test_df  = create_df_from_split("test")

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

class_labels = sorted(train_df['Label'].unique())
label2id = {label: i for i, label in enumerate(class_labels)}
id2label = {i: label for label, i in label2id.items()}
print("label2id:", label2id)

# -------------------------
# Load model dari HuggingFace pretrained
# -------------------------
image_processor = VideoMAEImageProcessor.from_pretrained(MODEL_CHECKPOINT)
model = VideoMAEForVideoClassification.from_pretrained(
    MODEL_CHECKPOINT,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,  # wajib karena num_labels beda dari pretrained
).to(device)

# -------------------------
# Transform config dari image_processor
# -------------------------
mean = image_processor.image_mean
std  = image_processor.image_std

if "shortest_edge" in image_processor.size:
    height = width = image_processor.size["shortest_edge"]
else:
    height = image_processor.size["height"]
    width  = image_processor.size["width"]
resize_to = (height, width)

num_frames_to_sample = model.config.num_frames
sample_rate   = 4
fps           = 30
clip_duration = num_frames_to_sample * sample_rate / fps

print(f"Frames: {num_frames_to_sample} | Clip duration: {clip_duration:.2f}s | Resize: {resize_to}")

train_transform = Compose([
    ApplyTransformToKey(key="video", transform=Compose([
        UniformTemporalSubsample(num_frames_to_sample),
        Lambda(lambda x: x / 255.0),
        Normalize(mean, std),
        RandomShortSideScale(min_size=256, max_size=320),
        RandomCrop(resize_to),
        RandomHorizontalFlip(p=0.5),
    ])),
])

val_transform = Compose([
    ApplyTransformToKey(key="video", transform=Compose([
        UniformTemporalSubsample(num_frames_to_sample),
        Lambda(lambda x: x / 255.0),
        Normalize(mean, std),
        Resize(resize_to),
    ])),
])

# -------------------------
# Dataset
# -------------------------
class CustomVideoDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe
    def __len__(self):
        return len(self.dataframe)
    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        return row['Path'], label2id[row['Label']]

def make_labeled_paths(df):
    ds = CustomVideoDataset(df)
    return [(p, {'label': l}) for p, l in ds]

train_dataset = pytorchvideo.data.LabeledVideoDataset(
    labeled_video_paths=make_labeled_paths(train_df),
    clip_sampler=make_clip_sampler("random", clip_duration),
    decode_audio=False,
    transform=train_transform,
)
val_dataset = pytorchvideo.data.LabeledVideoDataset(
    labeled_video_paths=make_labeled_paths(val_df),
    clip_sampler=make_clip_sampler("uniform", clip_duration),
    decode_audio=False,
    transform=val_transform,
)
test_dataset = pytorchvideo.data.LabeledVideoDataset(
    labeled_video_paths=make_labeled_paths(test_df),
    clip_sampler=make_clip_sampler("uniform", clip_duration),
    decode_audio=False,
    transform=val_transform,
)

# -------------------------
# Metrics
# -------------------------
accuracy_metric  = evaluate.load("accuracy")
f1_metric        = evaluate.load("f1")
recall_metric    = evaluate.load("recall")
precision_metric = evaluate.load("precision")

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=1)
    refs  = eval_pred.label_ids
    return {
        "accuracy" : accuracy_metric.compute( predictions=preds, references=refs)["accuracy"],
        "f1"       : f1_metric.compute(       predictions=preds, references=refs, average="weighted")["f1"],
        "recall"   : recall_metric.compute(   predictions=preds, references=refs, average="weighted")["recall"],
        "precision": precision_metric.compute(predictions=preds, references=refs, average="weighted")["precision"],
    }

def collate_fn(examples):
    pixel_values = torch.stack([e["video"].permute(1, 0, 2, 3) for e in examples])
    labels       = torch.tensor([e["label"] for e in examples])
    return {"pixel_values": pixel_values, "labels": labels}

# -------------------------
# Training
# -------------------------
num_epochs  = 50
batch_size  = 8
lr          = 1e-3

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_dir=LOG_DIR,
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    warmup_ratio=0.1,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
    report_to="none",
    max_steps=(len(train_df) // batch_size) * num_epochs,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=image_processor,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
)

print("\n=== Mulai Training ===")
train_results = trainer.train()

trainer.save_model(os.path.join(OUTPUT_DIR, "final_model"))
with open(os.path.join(OUTPUT_DIR, "train_results.json"), "w") as f:
    json.dump(train_results.metrics, f, indent=2)
print("Training selesai. Model disimpan di:", OUTPUT_DIR)

# -------------------------
# Evaluasi pada test set
# -------------------------
print("\n=== Evaluasi Test Set ===")
test_trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=image_processor,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
)
results = test_trainer.evaluate()
print(results)

with open(os.path.join(OUTPUT_DIR, "test_results.json"), "w") as f:
    json.dump(results, f, indent=2)

Device: cuda
Train: 321 | Val: 40 | Test: 42
label2id: {'1_mengangguk': 0, '2_mengangkat_tangan': 1, '3_menggunakan_hp': 2, '4_menopang_kepala': 3, '5_menunduk': 4}


Loading weights: 100%|██████████| 160/160 [00:00<00:00, 15438.33it/s]
[transformers] VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     | 
---------------------------------------------------------------------+------------+-
videomae.encoder.layer.{0...11}.attention.attention.v_bias           | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.weight          | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.bias                | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.weight    | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.weight              | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.bias          | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.weight        | UNEXPECTED | 
videomae.encoder.layer.{0...11}.attention.attention.q_bias           | UNEXPECT

Frames: 16 | Clip duration: 2.13s | Resize: (224, 224)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



=== Mulai Training ===


...

/home/coder/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.21it/s]
/home/coder/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.18it/s]
/home/coder/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_divisi

Training selesai. Model disimpan di: /home/coder/output_model/videomae_zoomodel

=== Evaluasi Test Set ===


/home/coder/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Training Loss,Validation Loss,Epoch,Accuracy,F1,Recall,Precision
No log,1.625674,0,0.234043,0.161940,0.234043,0.217651


{'eval_loss': 1.6256743669509888, 'eval_accuracy': 0.23404255319148937, 'eval_f1': 0.16193955356590742, 'eval_recall': 0.23404255319148937, 'eval_precision': 0.2176510546191397}


In [ ]:
# ============================================================
# VideoMAE Fine-tuning — HPC code-server version
# Diadaptasi dari kode kating, path & dependencies disesuaikan
# untuk lingkungan /home/coder (bukan Google Colab)
# ============================================================

# HAPUS: !pip install (jalankan manual di terminal dulu)
# pip install pytorchvideo transformers evaluate scikit-learn imageio

import sys, os, types, json
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Fix torchvision compatibility (sama seperti kode kating)
from torchvision.transforms.functional import rgb_to_grayscale
functional_tensor = types.ModuleType("torchvision.transforms.functional_tensor")
functional_tensor.rgb_to_grayscale = rgb_to_grayscale
sys.modules["torchvision.transforms.functional_tensor"] = functional_tensor

from pytorchvideo.data import LabeledVideoDataset, make_clip_sampler
from pytorchvideo.transforms import (
    ApplyTransformToKey, Normalize, RandomShortSideScale,
    ShortSideScale, UniformTemporalSubsample,
)
from torchvision.transforms import (
    Compose, Lambda, RandomCrop, RandomHorizontalFlip, Resize,
)
from transformers import (
    VideoMAEImageProcessor, VideoMAEForVideoClassification,
    TrainingArguments, Trainer,
)
import evaluate
import pytorchvideo.data

# -------------------------
# PATH CONFIG — sesuaikan di sini saja
# -------------------------
ROOT_PATH    = "/home/coder/data_skripsi/dataset_raffi"
OUTPUT_DIR   = "/home/coder/output_model/videomae_zoomodel_data_raffi"
LOG_DIR      = "/home/coder/output_model/videomae_zoomodel_data_raffi/logs"

# Pakai pretrained dari HuggingFace — tidak butuh checkpoint kating
MODEL_CHECKPOINT = "MCG-NJU/videomae-base"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR,    exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# -------------------------
# Load dataset dari folder struktur train/val/test
# -------------------------
def create_df_from_split(split):
    split_dir = os.path.join(ROOT_PATH, split)
    paths, labels = [], []
    for label_name in sorted(os.listdir(split_dir)):
        label_dir = os.path.join(split_dir, label_name)
        if not os.path.isdir(label_dir):
            continue
        for fname in os.listdir(label_dir):
            if fname.lower().endswith((".mp4", ".avi")):
                paths.append(os.path.join(label_dir, fname))
                labels.append(label_name)
    return pd.DataFrame({"Path": paths, "Label": labels})

train_df = create_df_from_split("train")
val_df   = create_df_from_split("val")
test_df  = create_df_from_split("test")

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

class_labels = sorted(train_df['Label'].unique())
label2id = {label: i for i, label in enumerate(class_labels)}
id2label = {i: label for label, i in label2id.items()}
print("label2id:", label2id)

# -------------------------
# Load model dari HuggingFace pretrained
# -------------------------
image_processor = VideoMAEImageProcessor.from_pretrained(MODEL_CHECKPOINT)
model = VideoMAEForVideoClassification.from_pretrained(
    MODEL_CHECKPOINT,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True,  # wajib karena num_labels beda dari pretrained
).to(device)

# -------------------------
# Transform config dari image_processor
# -------------------------
mean = image_processor.image_mean
std  = image_processor.image_std

if "shortest_edge" in image_processor.size:
    height = width = image_processor.size["shortest_edge"]
else:
    height = image_processor.size["height"]
    width  = image_processor.size["width"]
resize_to = (height, width)

num_frames_to_sample = model.config.num_frames
sample_rate   = 4
fps           = 30
clip_duration = num_frames_to_sample * sample_rate / fps

print(f"Frames: {num_frames_to_sample} | Clip duration: {clip_duration:.2f}s | Resize: {resize_to}")

train_transform = Compose([
    ApplyTransformToKey(key="video", transform=Compose([
        UniformTemporalSubsample(num_frames_to_sample),
        Lambda(lambda x: x / 255.0),
        Normalize(mean, std),
        RandomShortSideScale(min_size=256, max_size=320),
        RandomCrop(resize_to),
        RandomHorizontalFlip(p=0.5),
    ])),
])

val_transform = Compose([
    ApplyTransformToKey(key="video", transform=Compose([
        UniformTemporalSubsample(num_frames_to_sample),
        Lambda(lambda x: x / 255.0),
        Normalize(mean, std),
        Resize(resize_to),
    ])),
])

# -------------------------
# Dataset
# -------------------------
class CustomVideoDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe
    def __len__(self):
        return len(self.dataframe)
    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        return row['Path'], label2id[row['Label']]

def make_labeled_paths(df):
    ds = CustomVideoDataset(df)
    return [(p, {'label': l}) for p, l in ds]

train_dataset = pytorchvideo.data.LabeledVideoDataset(
    labeled_video_paths=make_labeled_paths(train_df),
    clip_sampler=make_clip_sampler("random", clip_duration),
    decode_audio=False,
    transform=train_transform,
)
val_dataset = pytorchvideo.data.LabeledVideoDataset(
    labeled_video_paths=make_labeled_paths(val_df),
    clip_sampler=make_clip_sampler("uniform", clip_duration),
    decode_audio=False,
    transform=val_transform,
)
test_dataset = pytorchvideo.data.LabeledVideoDataset(
    labeled_video_paths=make_labeled_paths(test_df),
    clip_sampler=make_clip_sampler("uniform", clip_duration),
    decode_audio=False,
    transform=val_transform,
)

# -------------------------
# Metrics
# -------------------------
accuracy_metric  = evaluate.load("accuracy")
f1_metric        = evaluate.load("f1")
recall_metric    = evaluate.load("recall")
precision_metric = evaluate.load("precision")

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=1)
    refs  = eval_pred.label_ids
    return {
        "accuracy" : accuracy_metric.compute( predictions=preds, references=refs)["accuracy"],
        "f1"       : f1_metric.compute(       predictions=preds, references=refs, average="weighted")["f1"],
        "recall"   : recall_metric.compute(   predictions=preds, references=refs, average="weighted")["recall"],
        "precision": precision_metric.compute(predictions=preds, references=refs, average="weighted")["precision"],
    }

def collate_fn(examples):
    pixel_values = torch.stack([e["video"].permute(1, 0, 2, 3) for e in examples])
    labels       = torch.tensor([e["label"] for e in examples])
    return {"pixel_values": pixel_values, "labels": labels}

# -------------------------
# Training
# -------------------------
num_epochs  = 50
batch_size  = 8
lr          = 1e-3

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_dir=LOG_DIR,
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    warmup_ratio=0.1,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
    report_to="none",
    max_steps=(len(train_df) // batch_size) * num_epochs,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=image_processor,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
)

print("\n=== Mulai Training ===")
train_results = trainer.train()

trainer.save_model(os.path.join(OUTPUT_DIR, "final_model"))
with open(os.path.join(OUTPUT_DIR, "train_results.json"), "w") as f:
    json.dump(train_results.metrics, f, indent=2)
print("Training selesai. Model disimpan di:", OUTPUT_DIR)

# -------------------------
# Evaluasi pada test set
# -------------------------
print("\n=== Evaluasi Test Set ===")
test_trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=image_processor,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
)
results = test_trainer.evaluate()
print(results)

with open(os.path.join(OUTPUT_DIR, "test_results.json"), "w") as f:
    json.dump(results, f, indent=2)

Device: cuda
Train: 1404 | Val: 172 | Test: 181
label2id: {'1_mengangguk': 0, '2_mengangkat_tangan': 1, '3_menggunakan_hp': 2, '4_menopang_kepala': 3, '5_menunduk': 4}


Loading weights: 100%|██████████| 160/160 [00:00<00:00, 19033.12it/s]
[transformers] VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base
Key                                                                  | Status     | 
---------------------------------------------------------------------+------------+-
videomae.encoder.layer.{0...11}.attention.attention.v_bias           | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.layernorm_before.weight          | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.bias                | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.attention.output.dense.weight    | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.output.dense.weight              | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.bias          | UNEXPECTED | 
decoder.decoder_layers.{0, 1, 2, 3}.intermediate.dense.weight        | UNEXPECTED | 
videomae.encoder.layer.{0...11}.attention.attention.q_bias           | UNEXPECT

Frames: 16 | Clip duration: 2.13s | Resize: (224, 224)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



=== Mulai Training ===


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

: 